In [1]:
# INDICATORE INTENSITA TURISTICA

from pathlib import Path
import pandas as pd
import duckdb

BASE_DIR = Path(r"../../")
RAW_DIR = BASE_DIR / "data" / "raw"

FILE_POP = RAW_DIR / "popolazione_residente.csv"

print("CSV utilizzato:")
print(f"  {FILE_POP}")

CSV utilizzato:
  ../../data/raw/popolazione_residente.csv


In [2]:
print(f"\n=== FILE: {FILE_POP.name} ===")
pop = pd.read_csv(FILE_POP)
pop["comune"] = pop["comune"].astype(str).str.strip().str.title()
n_anni = sorted(pop["anno"].unique())
n_comuni = pop["comune"].nunique()
print(f"righe sorgente: {len(pop)} | colonne: {list(pop.columns)}")
print(f"anni: {n_anni} | comuni distinti: {n_comuni} (su 377 attesi)")


=== FILE: popolazione_residente.csv ===
righe sorgente: 1508 | colonne: ['comune', 'popolazione_residente', 'anno']
anni: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)] | comuni distinti: 377 (su 377 attesi)


In [3]:
n_bad = pop["popolazione_residente"].isna().sum() + (pop["popolazione_residente"] <= 0).sum()
print(f"popolazione nulla o <=0: {n_bad}")

dup = pop.duplicated(subset=["comune", "anno"]).sum()
print(f"righe duplicate comune-anno: {dup}")

#controllo qualità deve uscire 0

popolazione nulla o <=0: 0
righe duplicate comune-anno: 0


In [4]:
COMUNE_VERIFICA = "Cagliari"
r = pop[pop["comune"] == COMUNE_VERIFICA].sort_values("anno")
print(f"\n--- Verifica: '{COMUNE_VERIFICA}' ---")
print(r.to_string(index=False))


--- Verifica: 'Cagliari' ---
  comune  popolazione_residente  anno
Cagliari                 149092  2022
Cagliari                 148296  2023
Cagliari                 147411  2024
Cagliari                 146692  2025


In [5]:
def chiave_comune(s):
    if pd.isna(s):
        return None
    apostrofo_tipografico = chr(0x2019)
    return str(s).strip().lower().replace(apostrofo_tipografico, "'")

pop["chiave_comune"] = pop["comune"].map(chiave_comune)

ANNI = [2022, 2023, 2024, 2025]
SOGLIA_MESI_MINIMI = 10

# --- pesi di stagionalita' da porti/aeroporti (usati per ridistribuire sia
#     presenze che arrivi con mese non disponibile) ---
FILE_PORTI_AEROPORTI = RAW_DIR / "porti_aeroporti.csv"
pa = pd.read_csv(FILE_PORTI_AEROPORTI)
pa["data"] = pd.to_datetime(pa["data"])
pa["anno"] = pa["data"].dt.year
pa["mese"] = pa["data"].dt.month
arrivi_mensili_pa = pa.groupby(["anno", "mese"], as_index=False)["arrivi"].sum()
arrivi_annuali_pa = arrivi_mensili_pa.groupby("anno")["arrivi"].transform("sum")
arrivi_mensili_pa["peso_mese"] = (arrivi_mensili_pa["arrivi"] / arrivi_annuali_pa).round(6)
print(f"=== FILE: {FILE_PORTI_AEROPORTI.name} ===")
print("Pesi mensili di stagionalita' caricati")

tabelle_anno = {}
for anno in ANNI:
    FILE_PRESENZE = RAW_DIR / f"csv_opendata_comuni_{anno}.csv"
    print(f"\n=== FILE: {FILE_PRESENZE.name} ===")
    df = pd.read_csv(FILE_PRESENZE, usecols=["comune", "mese", "presenze", "arrivi"])

    # righe con comune stesso non identificabile -> escluse
    mask_comune_ignoto = df["comune"].astype(str).str.strip().str.lower() == "non disponibile"
    n_escluse = mask_comune_ignoto.sum()
    presenze_escluse = df.loc[mask_comune_ignoto, "presenze"].sum()
    if n_escluse:
        print(f"  righe escluse (comune non identificabile): {n_escluse} | presenze coinvolte: "
              f"{presenze_escluse} ({presenze_escluse / df['presenze'].sum() * 100:.4f}% del totale {anno})")
    df = df[~mask_comune_ignoto].copy()

    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    df["chiave_comune"] = df["comune"].map(chiave_comune)

    # righe con mese non assegnabile -> non scartate, ridistribuite (sia presenze che arrivi)
    mask_non_disp = df["mese"].astype(str).str.strip().str.lower() == "non disponibile"
    non_allocabili = (
        df[mask_non_disp].groupby("chiave_comune", as_index=False)[["presenze", "arrivi"]]
        .sum()
        .rename(columns={"presenze": "presenze_non_allocabili", "arrivi": "arrivi_non_allocabili"})
    )

    df_mensile = df[~mask_non_disp].copy()
    df_mensile["mese"] = df_mensile["mese"].astype(int)
    agg = (
        df_mensile.groupby(["chiave_comune", "mese"], as_index=False)[["presenze", "arrivi"]]
        .sum()
        .rename(columns={"presenze": "presenze_mese", "arrivi": "arrivi_mese"})
    )

    pesi_anno = arrivi_mensili_pa[arrivi_mensili_pa["anno"] == anno][["mese", "peso_mese"]]
    redistrib = non_allocabili.merge(pesi_anno, how="cross")
    redistrib["presenze_redistribuite"] = (
        redistrib["presenze_non_allocabili"] * redistrib["peso_mese"]
    ).round(2)
    redistrib["arrivi_redistribuiti"] = (
        redistrib["arrivi_non_allocabili"] * redistrib["peso_mese"]
    ).round(2)
    redistrib = redistrib[["chiave_comune", "mese", "presenze_redistribuite", "arrivi_redistribuiti"]]

    m = agg.merge(redistrib, on=["chiave_comune", "mese"], how="outer")
    for col_base, col_red in [("presenze_mese", "presenze_redistribuite"),
                               ("arrivi_mese", "arrivi_redistribuiti")]:
        m[col_base] = m[col_base].fillna(0)
        m[col_red] = m[col_red].fillna(0)
        m[col_base] = (m[col_base] + m[col_red]).round(2)
    m = m.drop(columns=["presenze_redistribuite", "arrivi_redistribuiti"])
    m["anno"] = anno

    pop_anno = pop[pop["anno"] == anno][["chiave_comune", "comune", "popolazione_residente"]]
    m = m.merge(pop_anno, on="chiave_comune", how="left")
    m["popolazione_residente"] = m["popolazione_residente"].astype(int)

    m["intensita_turistica_presenze"] = (m["presenze_mese"] / m["popolazione_residente"]).round(4)
    m["intensita_turistica_arrivi_vs_residenti"] = (m["arrivi_mese"] / m["popolazione_residente"]).round(4)

    m["n_mesi_con_dati"] = m.groupby("comune")["mese"].transform("nunique").astype(int)
    m["copertura_sufficiente"] = m["n_mesi_con_dati"] >= SOGLIA_MESI_MINIMI

    out = (
        m[["comune", "anno", "mese", "popolazione_residente", "presenze_mese", "intensita_turistica_presenze",
           "arrivi_mese", "intensita_turistica_arrivi_vs_residenti",
           "n_mesi_con_dati", "copertura_sufficiente"]]
        .sort_values(["comune", "mese"])
        .reset_index(drop=True)
    )
    tabelle_anno[anno] = out
    print(f"{anno}: {out['comune'].nunique()} comuni | {len(out)} righe")

=== FILE: porti_aeroporti.csv ===
Pesi mensili di stagionalita' caricati

=== FILE: csv_opendata_comuni_2022.csv ===
  righe escluse (comune non identificabile): 249 | presenze coinvolte: 85723 (0.5231% del totale 2022)
2022: 189 comuni | 2168 righe

=== FILE: csv_opendata_comuni_2023.csv ===
  righe escluse (comune non identificabile): 9 | presenze coinvolte: 190 (0.0012% del totale 2023)
2023: 288 comuni | 3363 righe

=== FILE: csv_opendata_comuni_2024.csv ===
  righe escluse (comune non identificabile): 8 | presenze coinvolte: 34 (0.0002% del totale 2024)
2024: 307 comuni | 3649 righe

=== FILE: csv_opendata_comuni_2025.csv ===
  righe escluse (comune non identificabile): 11 | presenze coinvolte: 280 (0.0013% del totale 2025)
2025: 322 comuni | 3804 righe


In [6]:
print("\n--- Verifica Villasimius 2025 ---")
print(tabelle_anno[2025][tabelle_anno[2025]["comune"] == "Villasimius"].to_string(index=False))


--- Verifica Villasimius 2025 ---
     comune  anno  mese  popolazione_residente  presenze_mese  intensita_turistica_presenze  arrivi_mese  intensita_turistica_arrivi_vs_residenti  n_mesi_con_dati  copertura_sufficiente
Villasimius  2025     1                   3735          582.0                        0.1558        149.0                                   0.0399               12                   True
Villasimius  2025     2                   3735          824.0                        0.2206        255.0                                   0.0683               12                   True
Villasimius  2025     3                   3735          908.0                        0.2431        348.0                                   0.0932               12                   True
Villasimius  2025     4                   3735        16240.0                        4.3481       5040.0                                   1.3494               12                   True
Villasimius  2025     5            

- Perche' l'indicatore principale si basa sulle PRESENZE e non sugli ARRIVI:

le presenze contano le notti di pernottamento (il "carico" cumulato nel tempo su alloggi, servizi, infrastrutture), mentre gli arrivi contano solo il numero di check-in, senza distinguere chi resta 1 notte da chi ne resta 10. Per un indicatore di overtourism, il carico prolungato nel tempo e' l'informazione rilevante, non il semplice conteggio di persone
arrivate. 


E' anche la stessa base gia' usata nella Densita' Turistica, quindi resta l'indicatore coerente con gli altri.

**La colonna arrivi viene comunque calcolata come confronto complementare.**

In [ ]:
# ============================================================
# Salvataggio CSV per anno + scrittura nel database
# ============================================================
OUT_DIR = BASE_DIR / "data" / "indicatore_intensita_turistica"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"
DB_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

for anno in ANNI:
    out_anno = tabelle_anno[anno]
    percorso_out = OUT_DIR / f"intensita_turistica_{anno}.csv"
    out_anno.to_csv(percorso_out, index=False, encoding="utf-8-sig")
    print(f"{anno}: {out_anno['comune'].nunique()} comuni | {len(out_anno)} righe | salvato -> {percorso_out}")

df_complessivo = pd.concat(tabelle_anno.values(), ignore_index=True)
df_complessivo.to_csv(f"{OUT_DIR}/intensita_turistica_complessiva.csv", index=False, encoding="utf-8-sig")

intensita_completa = pd.concat(tabelle_anno.values(), ignore_index=True)
con.register("intensita_turistica_temp", intensita_completa)
con.execute("""
    CREATE OR REPLACE TABLE presentation.intensita_turistica AS
    SELECT * FROM intensita_turistica_temp
""")

verifica_db = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni
    FROM presentation.intensita_turistica
""").df()
print("\n=== SCRITTURA NEL DATABASE - presentation.intensita_turistica ===")
print(verifica_db.to_string(index=False))

print("\n=== VERIFICA DAL DB - Cagliari 2025 ===")
print(con.execute("""
    SELECT comune, anno, mese, intensita_turistica_presenze, intensita_turistica_arrivi_vs_residenti
    FROM presentation.intensita_turistica
    WHERE comune = 'Cagliari' AND anno = 2025
    ORDER BY mese
""").df().to_string(index=False))

2022: 189 comuni | 2168 righe | salvato -> ../../data/indicatore_intensita_turistica/intensita_turistica_2022.csv
2023: 288 comuni | 3363 righe | salvato -> ../../data/indicatore_intensita_turistica/intensita_turistica_2023.csv
2024: 307 comuni | 3649 righe | salvato -> ../../data/indicatore_intensita_turistica/intensita_turistica_2024.csv
2025: 322 comuni | 3804 righe | salvato -> ../../data/indicatore_intensita_turistica/intensita_turistica_2025.csv

=== SCRITTURA NEL DATABASE - presentation.intensita_turistica ===
 n_righe  n_comuni  n_anni
   12984       330       4

=== VERIFICA DAL DB - Cagliari 2025 ===
  comune  anno  mese  intensita_turistica_presenze  intensita_turistica_arrivi_vs_residenti
Cagliari  2025     1                        0.2422                                   0.1131
Cagliari  2025     2                        0.2725                                   0.1332
Cagliari  2025     3                        0.3395                                   0.1682
Cagliari  2025

In [28]:
con.close()
print("Connessione al database chiusa.")

Connessione al database chiusa.
